# External transfer: TCGA-trained model on a GEO cohort

This notebook trains an `E2MModel` on TCGA lung adenocarcinoma (`LUAD`) and applies it to **GSE31210** (Okayama et al.), an independent lung-adenocarcinoma microarray cohort the model has never seen, to demonstrate cross-cohort prediction end to end.

**Requirements:** network access and `GEOparse` for the GEO download:

```bash
pip install GEOparse
```

`inmoose` is optional, and only for the ComBat variant in section 5.

**Caveat:** GSE31210 is Affymetrix microarray data, on a different measurement scale from the TCGA STAR counts the model is trained on. Sections 1—4 predict straight across that gap, which shows the *mechanics* — align genes by symbol, then predict — and sets a floor. Section 5 adds the batch correction the manuscript applies (ComBat / rank normalization) and reports the integration and the metrics before and after.

In [ ]:
from pathlib import Path
import pandas as pd

from e2m import Dataset, E2MModel

DATA_DIR = Path("e2m_data")
RESULT_DIR = Path("results/external_gse31210")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Train the mutation model on TCGA-LUAD

This downloads and prepares TCGA-LUAD on the first run (cached under `DATA_DIR`) and fits the multitask network on every matched sample. TMB is not needed here, so it is skipped.

In [ ]:
tcga = Dataset.from_tcga(["LUAD"], data_dir=DATA_DIR, with_tmb=False)
model = E2MModel().fit(tcga)
len(tcga.expression), len(model.targets)

## 2. Download and prepare the external cohort

`GEOparse` downloads the series matrix and its platform annotation. We map probes to gene symbols (taking the first symbol of multi-mapping probes), average duplicate symbols, and return a samples-by-genes frame with the same orientation the model expects.

In [ ]:
def _symbol_column(annotation_table):
    for name in ("Gene Symbol", "Gene symbol", "GENE_SYMBOL", "Symbol", "gene_assignment"):
        if name in annotation_table.columns:
            return name
    raise ValueError(
        f"No gene-symbol column in the platform annotation: {list(annotation_table.columns)}"
    )


def load_gse31210(cache_dir):
    """Download GSE31210, map probes to symbols, return a samples-by-genes frame."""
    import GEOparse

    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    gse = GEOparse.get_GEO(geo="GSE31210", destdir=str(cache_dir), silent=True)
    probes = gse.pivot_samples("VALUE")  # probes x samples

    platform = list(gse.gpls.values())[0]
    annotation = platform.table.set_index("ID")
    symbols = annotation[_symbol_column(annotation)].reindex(probes.index).astype(str)
    symbols = symbols.str.split(r"\s*///\s*").str[0].str.strip()

    keep = symbols.notna() & ~symbols.isin({"", "nan", "---"})
    probes = probes.loc[keep]
    probes.index = symbols[keep].values
    expression = probes.apply(pd.to_numeric, errors="coerce").groupby(level=0).mean().T
    expression.index.name = "sample"
    return expression

In [ ]:
external = load_gse31210(DATA_DIR / "geo")
external.shape

## 3. Predict mutation probabilities on the external cohort

Input genes are aligned to the model's training features by symbol. We set `min_feature_overlap=0.0` because a microarray covers only part of the protein-coding transcriptome; missing features are filled with their training means. Because the scales differ and no batch correction is applied, read the numbers as a mechanics demonstration rather than a calibrated result.

In [ ]:
probabilities = model.predict(external, min_feature_overlap=0.0)
probabilities.to_csv(RESULT_DIR / "gse31210_probabilities.csv")
probabilities.iloc[:5, :8]

In [ ]:
probabilities.mean().sort_values(ascending=False).head(10)

## 4. Evaluate against the reported EGFR and KRAS status

GSE31210 reports a `gene alteration status` for every tumor, so two of the model's targets can be scored on this cohort. The field takes four values across the 226 tumors: `EGFR mutation +` (127), `EGFR/KRAS/ALK -` (68), `KRAS mutation +` (20), and `ALK-fusion +` (11). A tumor is EGFR-positive only under the first value and KRAS-positive only under the third; every other tested tumor is a negative for that gene.

A few limits worth keeping in mind:

- ALK is not scored. The reported event is a rearrangement, not a gene-level nonsilent mutation, so it is not one of the model's targets.
- The labels come from the study's clinical assay rather than MC3 calls, and that assay covered only these three genes. A negative here means "not detected by that assay", which is weaker than a negative MC3 call.
- EGFR is mutated in 56% of this cohort against roughly 14% in TCGA-LUAD, because GSE31210 is a Japanese, never-smoker-enriched series. Read the ranking metrics rather than the raw probabilities: the model learned the TCGA prevalence and is not calibrated to this one.

The 20 normal-lung samples in the series carry no alteration status and drop out of the intersection below.

In [ ]:
def load_alteration_status(cache_dir):
    """EGFR / KRAS status reported for each GSE31210 tumor, from the cached download."""
    import GEOparse

    gse = GEOparse.get_GEO(geo="GSE31210", destdir=str(Path(cache_dir)), silent=True)
    reported = {}
    for name, gsm in gse.gsms.items():
        for field in gsm.metadata.get("characteristics_ch1", []):
            key, _, value = field.partition(":")
            if key.strip() == "gene alteration status":
                reported[name] = value.strip()

    status = pd.Series(reported, name="alteration").sort_index()
    return pd.DataFrame(
        {
            "EGFR": (status == "EGFR mutation +").astype(int),
            "KRAS": (status == "KRAS mutation +").astype(int),
        },
        index=status.index,
    )


status = load_alteration_status(DATA_DIR / "geo")
status.sum()

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

from e2m.metrics import normalized_auprc


def score_targets(probabilities, targets=("EGFR", "KRAS")):
    """Ranking metrics for the reported targets, over the samples both frames cover."""
    scored = status.index.intersection(probabilities.index)
    rows = []
    for target in targets:
        truth = status.loc[scored, target]
        probability = probabilities.loc[scored, target]
        prevalence = float(truth.mean())
        auprc = float(average_precision_score(truth, probability))
        rows.append(
            {
                "target": target,
                "n_samples": len(scored),
                "n_positive": int(truth.sum()),
                "prevalence": prevalence,
                "auprc": auprc,
                "normalized_auprc": normalized_auprc(auprc, prevalence),
                "roc_auc": float(roc_auc_score(truth, probability)),
            }
        )
    return pd.DataFrame(rows).set_index("target")


external_metrics = score_targets(probabilities)
external_metrics.to_csv(RESULT_DIR / "gse31210_external_metrics.csv")
external_metrics.round(3)

`normalized_auprc` is the same quantity the cross-validation tables report, `(AUPRC - prevalence) / (1 - prevalence)`, where 0 is chance at the given prevalence and 1 is a perfect ranking. Threshold metrics such as accuracy, F1, and MCC are left out on purpose: a cutoff chosen at TCGA prevalence does not carry over to a cohort with four times the EGFR rate.

Whatever lands above chance here comes from expression alone, across a platform change and without batch correction, so treat it as a floor. Correcting the external cohort against the TCGA training set, as the manuscript does, is the next step if you want a number to quote.

## 5. Batch-correct the external cohort, then look again

Everything above fed microarray values to a model fitted on RNA-seq counts. The gene *ranking* within a sample survives that change reasonably well, but the values themselves do not, so the standardization inside `predict` is applied with the wrong means and spreads. Correcting the external cohort onto the training distribution first is what the manuscript does.

Two methods below:

- **rank** replaces each external value with the reference value at the same within-gene quantile. It assumes only that the ordering of samples within a gene carries over, which is what survives a platform change, so it is the safe default here.
- **combat** runs empirical-Bayes location/scale correction across both cohorts and keeps the external part. It pools information across genes, which helps when the external cohort is small. Needs `pip install inmoose`.

One caveat to state plainly: correcting a cohort *against the data the model was trained on* uses the training distribution to reshape the external values. That is the right move when the goal is to make a prediction, but it means the corrected result is no longer an untouched independent validation. The uncorrected numbers in section 4 stay in the notebook for that reason — they are the honest floor, and this section is the ceiling.

In [ ]:
import numpy as np
from scipy.stats import rankdata


def align_to_reference(external, reference, method="rank"):
    """Put an external cohort on the reference cohort's scale, gene by gene.

    Both frames are samples-by-genes. Correction is only defined where the two cohorts
    share a gene, so the result is restricted to shared columns; `predict` fills any
    remaining training feature with its training mean.
    """
    shared = [gene for gene in external.columns if gene in set(reference.columns)]
    if not shared:
        raise ValueError("The two matrices share no gene columns.")
    ext = external[shared].to_numpy(dtype=float)
    ref = reference[shared].to_numpy(dtype=float)

    if method == "rank":
        # Average ranks handle ties; the 0.5 offset keeps the extremes off the boundary.
        quantiles = (rankdata(ext, method="average", axis=0) - 0.5) / len(ext)
        ref_sorted = np.sort(ref, axis=0)
        position = quantiles * (len(ref) - 1)
        lower = np.floor(position).astype(int)
        upper = np.minimum(lower + 1, len(ref) - 1)
        fraction = position - lower
        columns = np.arange(len(shared))[None, :]
        corrected = (
            ref_sorted[lower, columns] * (1.0 - fraction)
            + ref_sorted[upper, columns] * fraction
        )
    elif method == "combat":
        from inmoose.pycombat import pycombat_norm

        combined = np.vstack([ref, ext]).T          # ComBat wants genes x samples
        batches = np.array([0] * len(ref) + [1] * len(ext))
        corrected = np.asarray(pycombat_norm(combined, batches)).T[len(ref):]
    else:
        raise ValueError(f"method must be 'rank' or 'combat', got {method!r}")

    return pd.DataFrame(corrected, index=external.index, columns=shared)


corrected = align_to_reference(external, tcga.expression, method="rank")
corrected.shape

### Integration before and after

PCA on the two cohorts stacked together, coloured by cohort. Before correction the platform is the dominant axis of variation and the cohorts sit apart; after correction they should overlap, which is the point of the exercise.

The first number in each title is the mean fraction of a sample's 30 nearest neighbours drawn from the *other* cohort, computed on the first 20 principal components. Near 0 means the cohorts occupy separate neighbourhoods. The second number is what fully mixed cohorts of these sizes would give: for a cohort holding a share `p` of the combined samples that value is `2p(1-p)`, well below 1 whenever the two cohorts differ in size, so compare against it rather than against 1.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler


def neighbour_mixing(values, labels, n_neighbors=30):
    """Mean fraction of each sample's neighbours that come from the other cohort."""
    finder = NearestNeighbors(n_neighbors=n_neighbors + 1).fit(values)
    neighbours = finder.kneighbors(values, return_distance=False)[:, 1:]  # drop self
    return float((labels[neighbours] != labels[:, None]).mean())


def integration_panel(reference, variants, n_neighbors=30):
    """PCA of reference + external, one panel per variant, coloured by cohort."""
    fig, axes = plt.subplots(1, len(variants), figsize=(5.4 * len(variants), 4.4), squeeze=False)
    for ax, (title, ext) in zip(axes[0], variants.items()):
        shared = [gene for gene in ext.columns if gene in set(reference.columns)]
        stacked = np.vstack([
            reference[shared].to_numpy(dtype=float),
            ext[shared].to_numpy(dtype=float),
        ])
        labels = np.array(["TCGA-LUAD"] * len(reference) + ["GSE31210"] * len(ext))

        scaled = StandardScaler().fit_transform(stacked)
        embedding = PCA(n_components=min(20, *scaled.shape) , random_state=0).fit_transform(scaled)

        for cohort, colour in (("TCGA-LUAD", "#4C72B0"), ("GSE31210", "#DD8452")):
            mask = labels == cohort
            ax.scatter(embedding[mask, 0], embedding[mask, 1], s=10, alpha=0.6,
                       c=colour, label=cohort, linewidths=0)
        mixing = neighbour_mixing(embedding, labels, n_neighbors)
        # With cohorts of unequal size, perfect mixing is 2p(1-p), not 1.
        share = len(ext) / len(labels)
        ax.set_title(f"{title}\nneighbour mixing {mixing:.2f} (balanced {2 * share * (1 - share):.2f})")
        ax.set_xlabel("PC-1")
        ax.set_ylabel("PC-2")
        ax.set_xticks([])
        ax.set_yticks([])
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
    axes[0][0].legend(frameon=False, loc="best", fontsize=9)
    fig.tight_layout()
    plt.show()


integration_panel(
    tcga.expression,
    {"Before correction": external, "After rank correction": corrected},
)

### Metrics before and after

Same labels and same model as section 4, with the corrected matrix as input.

In [ ]:
corrected_probabilities = model.predict(corrected, min_feature_overlap=0.0)
corrected_probabilities.to_csv(RESULT_DIR / "gse31210_probabilities_rank_corrected.csv")

comparison = pd.concat(
    {
        "uncorrected": external_metrics,
        "rank-corrected": score_targets(corrected_probabilities),
    },
    names=["input"],
)
comparison.to_csv(RESULT_DIR / "gse31210_metrics_before_after.csv")
comparison.round(3)

Swap `method="combat"` into the `align_to_reference` call above to compare the two corrections; pass both to `integration_panel` to see them side by side:

```python
combat_corrected = align_to_reference(external, tcga.expression, method="combat")
integration_panel(
    tcga.expression,
    {
        "Before correction": external,
        "After rank correction": corrected,
        "After ComBat": combat_corrected,
    },
)
```

Neither correction invents signal. If the corrected metrics barely move, the honest reading is that the shared genes carry little transferable signal for that target on this platform, not that the correction failed.